# AutoStrat quick test

Run setup once, edit the prompt, then run **Generate**. Only the accepted DSL and its Python execution wrapper appear; diagnostics are collapsed below. No microscope commands are executed.

Use the EvoMachine `.venv` notebook kernel with the matching AutoStrat checkout. The existing `explore_autostrat_pipeline.ipynb` remains available for deeper virtual-hardware tests.

In [1]:
import asyncio
import os
from getpass import getpass
from html import escape
from pathlib import Path

from IPython.display import Code, HTML, display
from autostrat import StrategyPipeline, load_domain_pack
from autostrat.generation import GeneratorConfig, PromptRecipe
from autostrat.verification import SemanticVerifierConfig
from evomachine.strategy_generation.preview import generate_preview

root = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "evomachine/domain_packs/microscopy/domain.yaml").is_file()
)
domain = load_domain_pack(root / "evomachine/domain_packs/microscopy")

# Same defaults as the existing test notebook; environment variables override them.
os.environ.setdefault("OPENAI_BASE_URL", "https://robin-office-2.tail32bb7.ts.net/v1")
model_id = os.getenv("AUTOSTRAT_MODEL_ID", "qwen3.6")
model = model_id if ":" in model_id else f"openai-chat:{model_id}"
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Model API key: ")

pipeline = StrategyPipeline(
    domain,
    generator_config=GeneratorConfig(model=model, validation_retries=2),
    verifier_config=SemanticVerifierConfig(model=model, output_retries=2),
    prompt_recipe=PromptRecipe(name="quick-test", few_shot_count=3),
    semantic_revisions=2,
)
print(f"Ready: {model} | microscopy {domain.metadata.version}")


/Users/liammetcalf/Code/workspace/evomachine/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
INFO:bfio.init:VERSION = 2.3.0

INFO:bfio.init:The bioformats_package.jar is not present.Can only use Python backend for reading/writing images.


Ready: openai-chat:qwen3.6 | microscopy 0.4.0


## Prompt

In [2]:
prompt = """
During initialisation, move to the first field of view.
At each step, terminate if 6 steps have completed.
Otherwise wait for (step_count + 1) / 2 seconds, bounded between 0.5 and 3 seconds.
During finalisation, move to the first field of view.
"""


## Generate

The Python output is the real EvoMachine wrapper, not a standalone translation of the DSL. The accepted DSL remains the source of behaviour. Model calls use the endpoint configured above.

In [4]:
# Clear the previous result before starting another request.
preview = None
preview = await asyncio.to_thread(generate_preview, pipeline, prompt)

if preview.verified is not None:
    display(HTML("<h3>DSL</h3>"))
    display(Code(preview.dsl, language="text"))
    display(HTML("<h3>Python — EvoMachine execution wrapper</h3>"))
    display(Code(preview.python, language="python"))
else:
    display(HTML("<b>No accepted strategy.</b><pre>" + escape(str(preview.error)) + "</pre>"))

display(HTML(
    "<details><summary>Diagnostics — attempts, revisions and errors</summary><pre>"
    + escape(preview.diagnostics())
    + "</pre></details>"
))


INFO:httpx2:HTTP Request: POST https://robin-office-2.tail32bb7.ts.net/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://robin-office-2.tail32bb7.ts.net/v1/chat/completions "HTTP/1.1 200 OK"


initialise
    move_fov(target=first_fov)
step
    const integer count = observation.step_count
    if count >= 6:
        terminate
    else:
        const number wait_time = max(0.5, min(3, (count + 1) / 2))
        wait(duration=wait_time)
finalise
    move_fov(target=first_fov)

from evomachine.strategy_generation import (
    AutoStratStrategy,
    MicroscopyCommandAdapter,
    MicroscopyObservationProvider,
    MicroscopyRuntimeErrorProvider,
)


def build_strategy(verified, domain, cfg):
    # verified is the accepted DSL result displayed above.
    # The shared evaluator handles expressions, conditions and recovery.
    return AutoStratStrategy(
        cfg=cfg,
        verified=verified,
        domain=domain,
        command_adapter=MicroscopyCommandAdapter(
            segment_images=False, save_images=False,
        ),
        observation_provider=MicroscopyObservationProvider(),
        runtime_error_provider=MicroscopyRuntimeErrorProvider(),
    )


# When ready, supply your application configuration:
# strategy = build_strategy(preview.verified, domain, cfg)
# Constructing a strategy does not start the microscope.

### Optional inspection

`preview.verified` holds the accepted result; `preview.attempts` holds semantic candidates and verdicts. For the full generation prompt, inspect `preview.verified.accepted.generated.prompt.messages` after success.

Successful-run deterministic retry counts are not currently exposed by AutoStrat and are reported as unavailable, not zero. This notebook checks generation and validation, not live observation values or hardware behaviour. Use the original notebook for virtual execution.